# 07 — Evaluate both translation systems (v2)

Scores BLEU and ChrF for the plain-source and entity-marked systems on the
22,059-sentence test set, runs a paired bootstrap, and breaks the result down by
entity type.

Markup comes from the corrected projection, not from the NER model, so these
figures bound from above what a recognizer-driven pipeline would deliver.

**Run from the repository root.** Kernel: `Python (tka)`.
Cell 3 reuses saved translations if it finds them, so a re-run is cheap.

## Cell 1: Setup

In [1]:
from pathlib import Path
import json, re, sys, time
import numpy as np
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print(f"Repo root: {ROOT}")

DATA   = ROOT / "data" / "processed" / "mt_v2"
MODELS = ROOT / "models"
RES    = ROOT / "results" / "mt"
FIGS   = ROOT / "paper" / "figures"
RES.mkdir(parents=True, exist_ok=True); FIGS.mkdir(parents=True, exist_ok=True)

SYSTEMS = {"baseline": (MODELS / "mt_baseline_v2", "mizo_plain"),
           "marked":   (MODELS / "mt_marked_v2",   "mizo_tagged")}

for name, (path, _) in SYSTEMS.items():
    ok = (path / "config.json").exists()
    print(("  ok   " if ok else "  MISS ") + f"{name}: {path.name}")
    if not ok:
        sys.exit("Run 06_mt_training_v2.ipynb first")

device = "cuda" if torch.cuda.is_available() else "cpu"
BEAM, GEN_BATCH, MAX_SRC, MAX_GEN = 4, 64, 128, 128
print(f"\nDevice {device}   beam {BEAM}   batch {GEN_BATCH}")

def write_json(obj, path, **kw):
    """Always UTF-8. Windows defaults to cp1252, which cannot encode the Mizo
    characters that appear in model output."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, **kw)

Repo root: C:\Users\Haulai\mizo-ner
  ok   baseline: mt_baseline_v2
  ok   marked: mt_marked_v2

Device cuda   beam 4   batch 64


## Cell 2: Test data

In [2]:
def lines(p):
    return open(p, encoding="utf-8").read().splitlines()

src_plain  = lines(DATA / "test_mizo_plain.txt")
src_tagged = lines(DATA / "test_mizo_tagged.txt")
refs       = lines(DATA / "test_english.txt")
assert len(src_plain) == len(src_tagged) == len(refs)
N = len(refs)
print(f"Test sentences: {N:,}")
print(f"\nplain : {src_plain[0]}")
print(f"tagged: {src_tagged[0]}")
print(f"ref   : {refs[0]}")

SRC = {"baseline": src_plain, "marked": src_tagged}

Test sentences: 22,059

plain : Tu nge a nih Kunga an a zawt.
tagged: Tu nge a nih «PERSON» Kunga «/PERSON» an a zawt.
ref   : Kunga asked who that was.


## Cell 3: Translate

Reuses `results/mt/hyps_*_v2.json` when present. Delete those files to force a
fresh run.

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def translate(model_dir, sources):
    tok = AutoTokenizer.from_pretrained(str(model_dir))
    mdl = AutoModelForSeq2SeqLM.from_pretrained(str(model_dir)).to(device).eval()
    order = sorted(range(len(sources)), key=lambda i: len(sources[i]))
    out = [None] * len(sources)
    t0 = time.time()
    for b in range(0, len(order), GEN_BATCH):
        ids = order[b:b + GEN_BATCH]
        enc = tok([sources[i] for i in ids], return_tensors="pt",
                  padding=True, truncation=True, max_length=MAX_SRC).to(device)
        with torch.no_grad():
            gen = mdl.generate(**enc, num_beams=BEAM, max_length=MAX_GEN)
        for i, g in zip(ids, tok.batch_decode(gen, skip_special_tokens=True)):
            out[i] = g.strip()
        if (b // GEN_BATCH) % 50 == 0:
            print(f"  {b + len(ids):,}/{len(sources):,}  "
                  f"{(time.time()-t0)/60:.1f} min", flush=True)
    print(f"  done in {(time.time()-t0)/60:.1f} min")
    del mdl; torch.cuda.empty_cache()
    assert all(o is not None for o in out)
    return out

hyps = {}
for name, (path, _) in SYSTEMS.items():
    cache = RES / f"hyps_{name}_v2.json"
    if cache.exists():
        hyps[name] = json.load(open(cache, encoding="utf-8"))
        if len(hyps[name]) == N:
            print(f"{name}: loaded {len(hyps[name]):,} cached translations")
            continue
        print(f"{name}: cache has {len(hyps[name]):,} lines, expected {N:,} - regenerating")
    print(f"\n--- {name} ---")
    hyps[name] = translate(path, SRC[name])
    write_json(hyps[name], cache)

for name in SYSTEMS:
    assert len(hyps[name]) == N
    print(f"\n{name}: {hyps[name][0]}")


--- baseline ---
  64/22,059  0.0 min
  3,264/22,059  0.3 min
  6,464/22,059  0.7 min
  9,664/22,059  1.1 min
  12,864/22,059  1.6 min
  16,064/22,059  2.2 min
  19,264/22,059  2.8 min
  done in 3.8 min
marked: loaded 22,059 cached translations

baseline: Kunga asked who it was.

marked: Kunga asked who it was.


## Cell 4: Corpus BLEU and ChrF

The sacrebleu signature can only be produced after scoring, since it records
the number of references, so it is read at the end of the loop.

In [11]:
import sacrebleu
from sacrebleu.metrics import BLEU, CHRF

bleu_metric, chrf_metric = BLEU(), CHRF()

scores = {}
for name in SYSTEMS:
    b = bleu_metric.corpus_score(hyps[name], [refs])
    c = chrf_metric.corpus_score(hyps[name], [refs])
    scores[name] = {"bleu": b.score, "chrf": c.score}
    print(f"{name:<10} BLEU {b.score:6.2f}   ChrF {c.score:6.2f}")

SIGNATURE = str(bleu_metric.get_signature())

d_bleu = scores["marked"]["bleu"] - scores["baseline"]["bleu"]
d_chrf = scores["marked"]["chrf"] - scores["baseline"]["chrf"]
print(f"\ndelta      BLEU {d_bleu:+6.2f}   ChrF {d_chrf:+6.2f}")
print(f"\nsignature: {SIGNATURE}")
print(f"previous run, corrupted source: BLEU 49.74 -> 50.06, delta +0.33")

baseline   BLEU  49.21   ChrF  68.39
marked     BLEU  49.53   ChrF  68.57

delta      BLEU  +0.33   ChrF  +0.18

signature: nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0
previous run, corrupted source: BLEU 49.74 -> 50.06, delta +0.33


## Cell 5: Paired bootstrap

1,000 resamples drawn with replacement. Both systems are scored on the same
resample each time, so the comparison is paired.

In [12]:
B = 1000
rng = np.random.default_rng(42)
diffs = np.empty(B)

t0 = time.time()
for k in range(B):
    idx = rng.integers(0, N, N)
    r  = [refs[i] for i in idx]
    hb = [hyps["baseline"][i] for i in idx]
    hm = [hyps["marked"][i] for i in idx]
    diffs[k] = (sacrebleu.corpus_bleu(hm, [r]).score
                - sacrebleu.corpus_bleu(hb, [r]).score)
    if (k + 1) % 100 == 0:
        print(f"  {k+1}/{B}  ({(time.time()-t0)/60:.1f} min)", flush=True)

wins   = int((diffs > 0).sum())
losses = int((diffs <= 0).sum())
p_val  = losses / B
lo, hi = np.percentile(diffs, [2.5, 97.5])

print(f"\nmean difference : {diffs.mean():+.3f} BLEU")
print(f"95% CI          : [{lo:+.3f}, {hi:+.3f}]")
print(f"marked higher   : {wins}/{B}")
print(f"baseline higher : {losses}/{B}")
if losses == 0:
    print(f"p               : < 0.001  (no resample favoured the baseline)")
else:
    print(f"p               : {p_val:.4f}")
if lo <= 0 <= hi:
    print("\nNote: the interval spans zero, so the direction of the effect is"
          "\nnot established at the 95% level.")

  100/1000  (7.7 min)
  200/1000  (15.2 min)
  300/1000  (22.8 min)
  400/1000  (30.4 min)
  500/1000  (37.8 min)
  600/1000  (45.3 min)
  700/1000  (53.0 min)
  800/1000  (60.7 min)
  900/1000  (68.4 min)
  1000/1000  (76.0 min)

mean difference : +0.327 BLEU
95% CI          : [+0.139, +0.527]
marked higher   : 1000/1000
baseline higher : 0/1000
p               : < 0.001  (no resample favoured the baseline)


## Cell 6: Sentence-level outcomes

In [13]:
from sacrebleu.metrics import BLEU as _BLEU
sent_bleu = _BLEU(effective_order=True)

better = worse = same = 0
for h_b, h_m, r in zip(hyps["baseline"], hyps["marked"], refs):
    if h_b == h_m:
        same += 1
        continue
    sb = sent_bleu.sentence_score(h_b, [r]).score
    sm = sent_bleu.sentence_score(h_m, [r]).score
    if sm > sb: better += 1
    elif sm < sb: worse += 1
    else: same += 1

print(f"{'Outcome':<24}{'Sentences':>11}{'Share':>9}")
print("-" * 44)
print(f"{'Marked better':<24}{better:>11,}{better/N*100:>8.1f}%")
print(f"{'Baseline better':<24}{worse:>11,}{worse/N*100:>8.1f}%")
print(f"{'No difference':<24}{same:>11,}{same/N*100:>8.1f}%")
print(f"\nnet margin: {better-worse:+,} sentences ({(better-worse)/N*100:+.2f}%)")

Outcome                   Sentences    Share
--------------------------------------------
Marked better                 4,707    21.3%
Baseline better               4,460    20.2%
No difference                12,892    58.4%

net margin: +247 sentences (+1.12%)


## Cell 7: By entity type

BLEU over the subset of test sentences containing at least one entity of each
type. A sentence with several types appears in several rows, so counts sum to
more than the test size.

In [14]:
TAG = re.compile("\u00ab([A-Z_]+)\u00bb")
types_per_sent = [set(TAG.findall(t)) for t in src_tagged]
all_types = sorted({t for s in types_per_sent for t in s})

rows = []
for lab in all_types:
    ids = [i for i, s in enumerate(types_per_sent) if lab in s]
    if len(ids) < 5:
        print(f"  skipping {lab}: only {len(ids)} sentences")
        continue
    r  = [refs[i] for i in ids]
    bb = sacrebleu.corpus_bleu([hyps["baseline"][i] for i in ids], [r]).score
    bm = sacrebleu.corpus_bleu([hyps["marked"][i]   for i in ids], [r]).score
    rows.append({"entity": lab, "sentences": len(ids),
                 "baseline": bb, "marked": bm, "delta": bm - bb})

rows.sort(key=lambda x: -x["sentences"])
print(f"\n{'Entity':<14}{'Sent':>8}{'Baseline':>10}{'Marked':>9}{'Delta':>8}")
print("-" * 49)
for r_ in rows:
    print(f"{r_['entity']:<14}{r_['sentences']:>8,}{r_['baseline']:>10.2f}"
          f"{r_['marked']:>9.2f}{r_['delta']:>+8.2f}")
print("-" * 49)
print(f"{'sum':<14}{sum(r_['sentences'] for r_ in rows):>8,}")


Entity            Sent  Baseline   Marked   Delta
-------------------------------------------------
PERSON          13,719     51.84    51.81   -0.02
GPE              5,077     49.48    50.09   +0.61
ORG              4,645     47.61    48.27   +0.66
NORP             1,000     47.17    47.62   +0.45
LOC                341     48.78    49.17   +0.40
LANGUAGE           226     48.92    48.03   -0.89
WORK_OF_ART        201     34.61    36.23   +1.62
PRODUCT            174     48.96    49.47   +0.51
FAC                166     52.41    54.32   +1.91
EVENT               31     39.44    45.38   +5.94
LAW                 21     54.94    54.28   -0.66
-------------------------------------------------
sum             25,601


## Cell 8: Save

In [15]:
out = {
    "test_sentences": N,
    "beam": BEAM,
    "sacrebleu_signature": SIGNATURE,
    "corpus": {k: {"bleu": round(v["bleu"], 2), "chrf": round(v["chrf"], 2)}
               for k, v in scores.items()},
    "delta_bleu": round(d_bleu, 3),
    "delta_chrf": round(d_chrf, 3),
    "bootstrap": {"replicates": B, "mean_diff": round(float(diffs.mean()), 3),
                  "ci_low": round(float(lo), 3), "ci_high": round(float(hi), 3),
                  "marked_wins": wins, "baseline_wins": losses,
                  "p_value": round(p_val, 4)},
    "sentence_level": {"marked_better": better, "baseline_better": worse,
                       "no_difference": same},
    "per_entity": [{k: (round(v, 2) if isinstance(v, float) else v)
                    for k, v in r_.items()} for r_ in rows],
    "previous_run": {"note": "corrupted source text",
                     "baseline_bleu": 49.74, "marked_bleu": 50.06,
                     "delta_bleu": 0.33, "delta_chrf": 0.24},
    "note": "Markup from the corrected projection, not from the NER model. "
            "Upper bound on a recognizer-driven pipeline.",
}
write_json(out, RES / "evaluation_v2.json", indent=2)
print(f"-> {(RES/'evaluation_v2.json').relative_to(ROOT)}")

-> results\mt\evaluation_v2.json


## Cell 9: Regenerate the per-type figure

In [16]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
matplotlib.rcParams.update({
    "font.family": "serif", "font.size": 9, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "figure.dpi": 300})
GREY = "0.45"

ent = [r_["entity"] for r_ in rows]
n_  = [r_["sentences"] for r_ in rows]
d_  = [r_["delta"] for r_ in rows]
THRESH = 900
split_at = sum(1 for v in n_ if v >= THRESH)
mean_d = float(diffs.mean())

fig, a = plt.subplots(figsize=(6.0, 2.7))
x = np.arange(len(ent))
a.bar(x, d_, width=0.62,
      color=["0.35" if v >= THRESH else "0.82" for v in n_],
      edgecolor="black", lw=0.7)
a.axhline(0, color="black", lw=0.8)
a.axhline(mean_d, color=GREY, ls=":", lw=1.0)
lim = max(abs(min(d_)), abs(max(d_)), abs(mean_d)) * 1.6
a.text(len(ent) - 0.4, mean_d + 0.035 * lim,
       f"corpus mean ${mean_d:+.2f}$", fontsize=7, color=GREY, ha="right")
if 0 < split_at < len(ent):
    a.axvline(split_at - 0.5, color=GREY, ls="--", lw=0.8)
for xi, (dv, nv) in enumerate(zip(d_, n_)):
    a.text(xi, dv + (0.04 * lim if dv >= 0 else -0.04 * lim), f"n={nv:,}",
           ha="center", va="bottom" if dv >= 0 else "top", fontsize=6.3)
a.set_xticks(x); a.set_xticklabels(ent, rotation=30, ha="right", fontsize=7.5)
a.set_ylabel(r"$\Delta$BLEU (marked $-$ baseline)")
a.set_ylim(-lim, lim)
a.grid(True, axis="y", alpha=0.22, lw=0.5); a.set_axisbelow(True)
for s in ("top", "right"): a.spines[s].set_visible(False)
fig.savefig(FIGS / "per_entity_delta.pdf"); plt.close(fig)
print("-> paper/figures/per_entity_delta.pdf")

-> paper/figures/per_entity_delta.pdf


## Cell 10: LaTeX rows

In [17]:
print("% ---- Table: corpus-level MT ----")
print(f"Baseline (plain source) & {scores['baseline']['bleu']:.2f} & "
      f"{scores['baseline']['chrf']:.2f} & --- & --- \\\\")
print(f"Entity-marked source    & \\textbf{{{scores['marked']['bleu']:.2f}}} & "
      f"\\textbf{{{scores['marked']['chrf']:.2f}}} & ${d_bleu:+.2f}$ & ${d_chrf:+.2f}$ \\\\")

print("\n% ---- Table: sentence level ----")
for lab, v in (("Marked system better", better), ("Baseline better", worse),
               ("No difference", same)):
    print(f"{lab:<22}& {v:,} & {v/N*100:.1f} \\\\")

print("\n% ---- Table: per entity ----")
for r_ in rows:
    esc = r_["entity"].replace("_", "\\_")
    print(f"{esc:<14}& {r_['sentences']:,} & {r_['baseline']:.2f} & "
          f"{r_['marked']:.2f} & ${r_['delta']:+.2f}$ \\\\")

pstr = "< 0.001" if losses == 0 else f"{p_val:.4f}"
print(f"\n% bootstrap: mean {diffs.mean():+.3f} BLEU, 95% CI "
      f"[{lo:+.3f}, {hi:+.3f}], {wins}/{B} resamples favour marked, p = {pstr}")
print(f"% sacrebleu: {SIGNATURE}")

% ---- Table: corpus-level MT ----
Baseline (plain source) & 49.21 & 68.39 & --- & --- \\
Entity-marked source    & \textbf{49.53} & \textbf{68.57} & $+0.33$ & $+0.18$ \\

% ---- Table: sentence level ----
Marked system better  & 4,707 & 21.3 \\
Baseline better       & 4,460 & 20.2 \\
No difference         & 12,892 & 58.4 \\

% ---- Table: per entity ----
PERSON        & 13,719 & 51.84 & 51.81 & $-0.02$ \\
GPE           & 5,077 & 49.48 & 50.09 & $+0.61$ \\
ORG           & 4,645 & 47.61 & 48.27 & $+0.66$ \\
NORP          & 1,000 & 47.17 & 47.62 & $+0.45$ \\
LOC           & 341 & 48.78 & 49.17 & $+0.40$ \\
LANGUAGE      & 226 & 48.92 & 48.03 & $-0.89$ \\
WORK\_OF\_ART & 201 & 34.61 & 36.23 & $+1.62$ \\
PRODUCT       & 174 & 48.96 & 49.47 & $+0.51$ \\
FAC           & 166 & 52.41 & 54.32 & $+1.91$ \\
EVENT         & 31 & 39.44 & 45.38 & $+5.94$ \\
LAW           & 21 & 54.94 & 54.28 & $-0.66$ \\

% bootstrap: mean +0.327 BLEU, 95% CI [+0.139, +0.527], 1000/1000 resamples favour marked, p = 